# MouserCV Frame Slicer — Extract & View Video Segments

Utility notebook for extracting frame ranges from any video in the
Bauer lab dataset. Useful for inspecting specific time windows,
finding interesting behaviors, and exporting frames for annotation.

**Usage:** Set `VIDEO_PATH`, `START_TIME`, `END_TIME` in the config cell, then Run All.

In [ ]:
# === CONFIGURATION ===
import os

# Path to video file
VIDEO_PATH = "../data/videos/CV analysis project/Cage 2.MOV"

# Time range to extract (use the helper below to convert mm:ss to frames)
START_TIME = "1:30"   # mm:ss or mm:ss.ms
END_TIME = "2:00"     # mm:ss or mm:ss.ms

# How many sample frames to display
NUM_DISPLAY_FRAMES = 8

# Output directory for extracted frames
OUTPUT_DIR = "../data/frames_cache"

In [ ]:
def time_to_seconds(time_str: str) -> float:
    """Convert mm:ss or mm:ss.ms or hh:mm:ss to seconds."""
    parts = time_str.strip().split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return float(time_str)


def seconds_to_time(seconds: float) -> str:
    """Convert seconds to mm:ss.ms format."""
    m, s = divmod(seconds, 60)
    return f"{int(m)}:{s:05.2f}"


def time_to_frame(time_str: str, fps: float) -> int:
    """Convert mm:ss to frame number."""
    return int(time_to_seconds(time_str) * fps)


def frame_to_time(frame: int, fps: float) -> str:
    """Convert frame number to mm:ss."""
    return seconds_to_time(frame / fps)


# Quick reference table
print("Time -> Frame converter (assuming 30fps):")
for t in ["0:00", "0:30", "1:00", "2:00", "5:00", "10:00", "15:00"]:
    print(f"  {t} -> frame {time_to_frame(t, 30)}")

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = total_frames / fps

print(f"Video: {os.path.basename(VIDEO_PATH)}")
print(f"Resolution: {width}x{height}")
print(f"FPS: {fps}")
print(f"Total frames: {total_frames}")
print(f"Duration: {seconds_to_time(duration_sec)} ({duration_sec:.1f}s)")
print()

start_frame = time_to_frame(START_TIME, fps)
end_frame = time_to_frame(END_TIME, fps)
num_frames = end_frame - start_frame
print(f"Selected range: {START_TIME} -> {END_TIME}")
print(f"Frame range: {start_frame} -> {end_frame} ({num_frames} frames, {num_frames/fps:.1f}s)")
cap.release()

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

frames = []
for i in range(num_frames):
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()

print(f"Extracted {len(frames)} frames")

In [ ]:
import matplotlib.pyplot as plt

# Pick evenly spaced frames to display
indices = np.linspace(0, len(frames) - 1, NUM_DISPLAY_FRAMES, dtype=int)

cols = 4
rows = (NUM_DISPLAY_FRAMES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
axes = axes.flatten() if NUM_DISPLAY_FRAMES > 1 else [axes]

for ax_idx, frame_idx in enumerate(indices):
    frame_rgb = cv2.cvtColor(frames[frame_idx], cv2.COLOR_BGR2RGB)
    axes[ax_idx].imshow(frame_rgb)
    abs_frame = start_frame + frame_idx
    axes[ax_idx].set_title(
        f"Frame {abs_frame} ({frame_to_time(abs_frame, fps)})", fontsize=10
    )
    axes[ax_idx].axis("off")

# Hide unused axes
for ax_idx in range(len(indices), len(axes)):
    axes[ax_idx].axis("off")

plt.suptitle(
    f"{os.path.basename(VIDEO_PATH)} — {START_TIME} to {END_TIME}",
    fontsize=14,
)
plt.tight_layout()
plt.show()

In [ ]:
# Uncomment to save frames to disk
# video_name = os.path.splitext(os.path.basename(VIDEO_PATH))[0].replace(" ", "_")
# save_dir = os.path.join(OUTPUT_DIR, video_name)
# os.makedirs(save_dir, exist_ok=True)
# for i, frame in enumerate(frames):
#     frame_num = start_frame + i
#     cv2.imwrite(os.path.join(save_dir, f"frame_{frame_num:06d}.jpg"), frame)
# print(f"Saved {len(frames)} frames to {save_dir}")

## Video Index

| # | File | Res | FPS | Duration | Mice |
|---|------|-----|-----|----------|------|
| 1 | Before sacri Regina.MOV | 1080p | 30 | 16:11 | 2 |
| 2 | Cage 17082 video.MOV | 1080p | 30 | 17:22 | 3 |
| 3 | Cage 17083 video.MOV | 4K | 24 | 15:08 | 2 |
| 4 | Cage 2.MOV | 1080p | 30 | 17:21 | 2 |
| 5 | Female mut...treatment.MOV | 4K | 24 | 17:02 | 1-2 |
| 6 | IMG_5850.MOV | 4K | 24 | 7:39 | 3 |
| 7 | DOB 160810 +.AVI | 320x216 | 24 | 5:37 | 1 |